In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# ---------------- Z-SKORLU VE DİNAMİK F1-OPTİMİZASYONLU ULTIMATE XGBOOST ----------------

%pip install xgboost pyod scikit-learn -q

import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import precision_recall_curve, auc, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("1. Veri yükleniyor ve temizleniyor...")
file_name = "/content/drive/MyDrive/financial_anomaly_benchmark_data (1).csv" # Kendi yolun
df = pd.read_csv(file_name)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.ffill().fillna(0)

print("2. Üst Düzey Özellik Mühendisliği (Z-Skorları) Uygulanıyor...")
# 1. Z-Skorları (Anormalliği İstatistiksel Olarak Ölçme)
rolling_vol = df['Volume_Change'].abs().rolling(window=20)
df['Volume_Z_Score'] = (df['Volume_Change'].abs() - rolling_vol.mean()) / (rolling_vol.std() + 1e-8)

rolling_vola = df['Volatility_HighLow'].rolling(window=20)
df['Vola_Z_Score'] = (df['Volatility_HighLow'] - rolling_vola.mean()) / (rolling_vola.std() + 1e-8)

# 2. RSI ve Bollinger
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).ewm(alpha=1/14, adjust=False).mean()
loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/14, adjust=False).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

std = df['Close'].rolling(window=20).std()
ma = df['Close'].rolling(window=20).mean()
df['BB_Width'] = (std * 4) / ma

df = df.fillna(0)

# Gerçek Anomali Etiketleri (%1) - Hedefimiz
df['Shock_Score'] = df['Volatility_HighLow'] * df['Volume_Change'].abs()
threshold = df['Shock_Score'].quantile(0.99)
df['y_true'] = (df['Shock_Score'] >= threshold).astype(int)

# Girdi Özellikleri (Sadece en güçlü istatistikleri bıraktık)
features = ['Returns', 'Volume_Z_Score', 'Vola_Z_Score', 'RSI', 'BB_Width']

# 🚨 Gelecek Tahmini İçin (Sızıntı Önleme) 1 Periyot Geriye Kaydırıyoruz 🚨
X = df[features].shift(1).fillna(0)
y = df['y_true']

# KAYAN PENCERE AYARLARI
train_window_size = 2880
test_window_size = 96
test_indices = np.arange(train_window_size, len(df), test_window_size)

y_test_real = []
y_test_scores = []
y_test_preds = []

print(f"3. F1-Optimizasyonlu XGBoost Eğitimi Başlatılıyor ({len(test_indices)} döngü)...")
start_time = time.time()

for start_test_idx in tqdm(test_indices, desc="Ultimate XGBoost"):

    start_train_idx = start_test_idx - train_window_size
    end_test_idx = min(start_test_idx + test_window_size, len(df))

    X_train_window = X.iloc[start_train_idx:start_test_idx].copy()
    y_train_window = y.iloc[start_train_idx:start_test_idx].copy()

    X_test_window = X.iloc[start_test_idx:end_test_idx].copy()
    y_test_window = y.iloc[start_test_idx:end_test_idx].copy()

    if y_train_window.sum() == 0:
        y_test_real.extend(y_test_window.values)
        y_test_scores.extend(np.zeros(len(y_test_window)))
        y_test_preds.extend(np.zeros(len(y_test_window)))
        continue

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_window)
    X_test_scaled = scaler.transform(X_test_window)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(X_train_scaled)
    train_distances = np.min(kmeans.transform(X_train_scaled), axis=1)
    test_distances = np.min(kmeans.transform(X_test_scaled), axis=1)

    X_train_final = np.column_stack((X_train_scaled, train_distances))
    X_test_final = np.column_stack((X_test_scaled, test_distances))

    weight = len(y_train_window[y_train_window == 0]) / y_train_window.sum()

    model_xgb = xgb.XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        scale_pos_weight=weight,
        random_state=42,
        n_jobs=-1
    )

    model_xgb.fit(X_train_final, y_train_window)

    # 💡 İŞTE F1 SKORUNU PATLATACAK OPTİMİZASYON ADIMI 💡
    train_probs = model_xgb.predict_proba(X_train_final)[:, 1]

    # Eğitim verisi üzerinde tüm olası eşikleri test et
    precisions, recalls, thresholds_curve = precision_recall_curve(y_train_window, train_probs)

    # Sıfıra bölme hatasını önlemek için ufak bir epsilon (1e-8) ekliyoruz
    f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

    # F1 skorunu en yüksek yapan eşiği (threshold) bul!
    best_threshold_idx = np.argmax(f1_scores_curve)
    best_threshold = thresholds_curve[best_threshold_idx] if best_threshold_idx < len(thresholds_curve) else 0.5

    # Test verisinde bu "En İyi F1 Eşiğini" kullan
    test_probs = model_xgb.predict_proba(X_test_final)[:, 1]
    test_preds = (test_probs >= best_threshold).astype(int)

    y_test_real.extend(y_test_window.values)
    y_test_scores.extend(test_probs)
    y_test_preds.extend(test_preds)

end_time = time.time()
inference_time_ms = ((end_time - start_time) / len(test_indices)) * 1000

print("\n4. Final Başarı Metrikleri Hesaplanıyor...")
roc_auc = roc_auc_score(y_test_real, y_test_scores)
precision, recall, _ = precision_recall_curve(y_test_real, y_test_scores)
final_pr_auc = auc(recall, precision)
final_f1 = f1_score(y_test_real, y_test_preds)

print("\n" + "═"*65)
print(" 🚀 ULTIMATE XGBOOST (Z-SCORE + DİNAMİK F1 OPT.) SONUÇLARI 🚀")
print("═"*65)
print(f"ROC-AUC Skoru                : {roc_auc:.4f}")
print(f"PR-AUC Skoru                 : {final_pr_auc:.4f}")
print(f"F1 Skoru                     : {final_f1:.4f} (Matematiksel olarak zorlanmış en iyi değer)")
print(f"Ort. Çıkarım Süresi (Pencere): {inference_time_ms:.2f} milisaniye")
print("═"*65)

results_df = pd.DataFrame({
    'Model': ['Ultimate XGBoost (F1 Optimized)'],
    'ROC_AUC': [roc_auc],
    'PR_AUC': [final_pr_auc],
    'F1_Score': [final_f1],
    'Inference_Time_ms': [inference_time_ms]
})
results_df.to_csv('/content/drive/MyDrive/benchmark_results_3.csv', mode='w', header=True, index=False)
print("✅ Nihai sonuçlar CSV'ye eklendi!")

1. Veri yükleniyor ve temizleniyor...
2. Üst Düzey Özellik Mühendisliği (Z-Skorları) Uygulanıyor...
3. F1-Optimizasyonlu XGBoost Eğitimi Başlatılıyor (1439 döngü)...


Ultimate XGBoost: 100%|██████████| 1439/1439 [03:23<00:00,  7.07it/s]



4. Final Başarı Metrikleri Hesaplanıyor...

═════════════════════════════════════════════════════════════════
 🚀 ULTIMATE XGBOOST (Z-SCORE + DİNAMİK F1 OPT.) SONUÇLARI 🚀
═════════════════════════════════════════════════════════════════
ROC-AUC Skoru                : 0.5766
PR-AUC Skoru                 : 0.0125
F1 Skoru                     : 0.0129 (Matematiksel olarak zorlanmış en iyi değer)
Ort. Çıkarım Süresi (Pencere): 141.35 milisaniye
═════════════════════════════════════════════════════════════════
✅ Nihai sonuçlar CSV'ye eklendi!
